# 一、数据来源

## 1、测试样本（HG002 BAM）

HG002 样本 60x 覆盖度的 Illumina HiSeq 短读长比对文件 。
下载指令：

In [ ]:
wget ftp://ftp-trace.ncbi.nlm.nih.gov/giab/ftp/data/AshkenazimTrio/HG002_NA24385_son/NIST_HiSeq_HG002_Homogeneity-10953946/NHGRI_Illumina300X_AJtrio_novoalign_bams/HG002.hs37d5.60x.1.bam

In [ ]:
wget ftp://ftp-trace.ncbi.nlm.nih.gov/giab/ftp/data/AshkenazimTrio/HG002_NA24385_son/NIST_HiSeq_HG002_Homogeneity-10953946/NHGRI_Illumina300X_AJtrio_novoalign_bams/HG002.hs37d5.60x.1.bam.bai

## 2、参考基因组

原论文 HG002 评测基于 GRCh37 (hg19) 版本的参考序列 。

In [ ]:
wget ftp://ftp.1000genomes.ebi.ac.uk/vol1/ftp/technical/reference/phase2_reference_assembly_sequence/hs37d5.fa.gz
gunzip hs37d5.fa.gz
samtools faidx hs37d5.fa

## 3、金标准文件与高置信度区域

原论文明确使用 GIAB NIST Tier 1 v0.6 版本的金标准 ，且仅评估该版本提供的 2.51 Gb 高置信度区域内的变异 。

In [ ]:
# 下载金标准 VCF
wget ftp://ftp-trace.ncbi.nlm.nih.gov/giab/ftp/data/AshkenazimTrio/analysis/NIST_SVs_Integration_v0.6/HG002_SVs_Tier1_v0.6.vcf.gz

In [ ]:
# 下载考试范围 BED
wget ftp://ftp-trace.ncbi.nlm.nih.gov/giab/ftp/data/AshkenazimTrio/analysis/NIST_SVs_Integration_v0.6/HG002_SVs_Tier1_v0.6.bed

# 二、环境搭建

Cue 官方基于 Python 3.7 和 PyTorch 1.5.1 开发 。

在VCF处理阶段建议仅使用Linux原生命令，来避免如bcftools等工具触发的libcrypto动态库依赖崩溃报错。

In [ ]:
conda create -n cue_env python=3.7 -y
conda activate cue_env

pip install torch==1.5.1 torchvision==0.6.1

conda install -c bioconda htslib -y
conda install -c bioconda bcftools -y
conda install -c conda-forge gsl=2.5 -y

git clone https://github.com/popiclab/cue.git cue-master
cd cue-master
conda install -c conda-forge pycocotools -y  #使用 Conda 预编译通道安装 pycocotools，绕开本地 gcc 编译报错


pip install -r install/requirements.txt
rm -rf /mnt/home/ygjx/chenkejin/anaconda3/envs/cue_env/lib/python3.7/site-packages/numpy*
pip install numpy==1.21.6

export PYTHONPATH=${PYTHONPATH}:$(pwd)

pip install truvari==3.2.0

mkdir -p data/models
wget --directory-prefix=data/models/ https://storage.googleapis.com/cue-models/latest/cue.v2.pt
    
    
# 1. 创建新环境并直接安装最新版的 truvari
conda create -n truvari_v4 -c conda-forge -c bioconda truvari -y

# 2. 激活新环境
conda activate truvari_v4

# 3. 验证版本（应该显示 v4.x.x）
truvari version

# 三、运行过程

## 1、构建配置文件

In [ ]:
#创建config/hg002_data.yaml文件并写入下面内容：
bam: "/chenkejin/cue-master/data/hg002_benchmark/HG002.hs37d5.60x.1.bam"
fai: "/chenkejin/cue-master/data/hg002_benchmark/hs37d5.fa.fai"
chr_names: null
#创建config/hg002_model.yaml文件，并写入下面内容：
model_path: "data/models/cue.v2.pt"
out_dir: "output_hg002/"
gpu_ids: [0]
n_cpus: 4
batch_size: 12

## 2、模型推理

In [ ]:
#导出环境变量
export PYTHONPATH=${PYTHONPATH}:$(pwd)
# 前台测试运行（看一眼有没有报错立刻停）
python engine/call.py --data_config config/hg002_data.yaml --model_config config/hg002_model.yaml
# 后台挂起运行（正式跑）
nohup python engine/call.py --data_config config/hg002_data.yaml --model_config config/hg002_model.yaml > cue_run.log 2>&1 &
#可使用tail -f cue_run.log 查看进度，结果默认输出在 config/reports/svs.vcf

## 3、VCF清洗

In [ ]:
# 1. 创建结果输出目录
mkdir -p output_hg002
cp config/reports/svs.vcf output_hg002/hg002_raw.vcf
cd output_hg002

# 2. 提取表头并对变异记录按染色体及坐标进行严格排序
(grep "^#" hg002_raw.vcf; grep -v "^#" hg002_raw.vcf | sort -k1,1V -k2,2n) > hg002_sorted.vcf

# 3. 标准化 bgzip 压缩与建库 (Truvari 评测的必须前置条件)
bgzip -c hg002_sorted.vcf > hg002_final.vcf.gz
tabix -p vcf hg002_final.vcf.gz
cd ..

## 4、金标准预处理

原论文仅针对 大于 5kb 的 DEL (缺失) 进行了深度评估，共有 138 个事件 。

In [ ]:
# 使用 zgrep 提取纯 DEL 金标准 (完美绕过依赖报错)
zgrep "^#" data/hg002_benchmark/HG002_SVs_Tier1_v0.6.vcf.gz > HG002_SVs_Tier1_v0.6_DEL_only.vcf
zgrep -v "^#" data/hg002_benchmark/HG002_SVs_Tier1_v0.6.vcf.gz | grep "SVTYPE=DEL" >> HG002_SVs_Tier1_v0.6_DEL_only.vcf

# 压缩并建索引
bgzip -c HG002_SVs_Tier1_v0.6_DEL_only.vcf > HG002_SVs_Tier1_v0.6_DEL_only.vcf.gz
tabix -p vcf HG002_SVs_Tier1_v0.6_DEL_only.vcf.gz

## 5、评估

In [ ]:
# 运行 Truvari 进行精确打分
truvari bench -b HG002_SVs_Tier1_v0.6_DEL_only.vcf.gz \
              -c output_hg002/hg002_final.vcf.gz \
              -o truvari_results_strict \
              --includebed HG002_SVs_Tier1_v0.6.bed \
              --passonly --refdist 500 --pctsize 0.5 --pctovl 0.5 --pctsim 0 \
              --sizemin 5000 --sizemax 10000000

# 打印最终成绩单
cat truvari_results_strict/summary.txt

# 四、462745N、462745T测试

# 1、数据来源

In [9]:
#A3服务器路径：bam: "/data/share/PancreaticWGS/462745T/462745T.sorted.markdup.BQSR.bam"
#bai："/data/share/PancreaticWGS/462745T/462745T.sorted.markdup.BQSR.bai"
#bam: "/data/share/PancreaticWGS/462745N/462745N.sorted.markdup.BQSR.bam"
#bai: "/data/share/PancreaticWGS/462745N/462745N.sorted.markdup.BQSR.bai"
#reference: "/data/share/PancreaticWGS/Homo_sapiens_assembly38.fasta.fai"

## 2、创建配置文件

cue-master/config/462745N_data.yaml

In [ ]:
bam: "/chenkejin/cue-master/data/462745/462745N/462745N.sorted.markdup.BQSR.bam"
fai: "/chenkejin/cue-master/data/462745/Homo_sapiens_assembly38.fasta.fai"
chr_names: null

cue-master/config/462745N_model.yaml

In [ ]:
model_path: "data/models/cue.v2.pt"
out_dir: "output_462745N/"
gpu_ids: [0]
n_cpus: 10
batch_size: 16

# 3、运行

In [ ]:
nohup python engine/call.py --data_config config/hg002_data.yaml --model_config config/hg002_model.yaml > cue_run.log 2>&1 &

# 4、VCF清洗

In [12]:
mkdir -p output_hg002
cp config/reports/svs.vcf output_462745N/462745N_raw.vcf   #不知道为什么，输出的vcf文件总是在config/reports/svs.vcf
cd output_462745N

(grep "^#" 462745N_raw.vcf; grep -v "^#" 462745N_raw.vcf | sort -k1,1V -k2,2n) > 462745N_sorted.vcf

bgzip -c 462745N_sorted.vcf > 462745N_final.vcf.gz
tabix -p vcf 462745N_final.vcf.gz
cd ..

# 5、金标准预处理（还未进行，462745没有金标准）

# 五、HCC1395

配置环境、运行与最上方的一样，配置文件改改参数就行。注意：A3服务器的bam文件时间戳比相应的bai文件新，且cue生成的一大堆中间文件会自动存储在bam文件目录下，因此需要新建一个文件夹为bam、bai文件创建软链接。然后修改yaml文件里的bam输入路径。

In [ ]:
ln -s /data/share/Genomics_datasets/HCC1395/WGS/WGS_EA_N_1.bwa.dedup.bam ./my_local_WGS_EA_N_1.bam
cp /data/share/Genomics_datasets/HCC1395/WGS/WGS_EA_N_1.bwa.dedup.bam.bai ./my_local_WGS_EA_N_1.bam.bai

N和T数据先分别经过Cue获得vcf结果文件。新建个环境来跑truvari

In [ ]:
# 1. 创建新环境并直接安装最新版的 truvari
conda create -n truvari_v4 -c conda-forge -c bioconda truvari -y

# 2. 激活新环境
conda activate truvari_v4

# 3. 验证版本（应该显示 v4.x.x）
truvari version

利用truvari来获得肿瘤特异性变异，-b设置为正常数据，-c设置为肿瘤数据

In [ ]:
truvari bench \
  -b "/data/chenkejin/cue-master/output_WGS_EA_N_1/WGS_EA_N_1_final.vcf.gz" \
  -c "/data/chenkejin/cue-master/output_WGS_EA_T_1/WGS_EA_T_1_final.vcf.gz" \
  -o "/data/chenkejin/cue-master/result/WGS_EA_1/T-N-4000" \
  -f /data/share/Genomics_datasets/HCC1395/reference_genome/GRCh38/GRCh38.d1.vd1.fa \
  --refdist 500 \
  --pctseq 0 \
  --pctsize 0.7 \
  --pctovl 0 \
  --sizemin 4000 \
  --sizemax -1 \
  --bnddist 100

对金标准文件进行处理

In [ ]:
cd /data/chenkejin/cue-master/

(grep "^#" High_Confidence_Somatic_SV_v1.2_FINAL.vcf; grep -v "^#" High_Confidence_Somatic_SV_v1.2_FINAL.vcf | sort -k1,1V -k2,2n) > gold_standard_sorted.vcf

bgzip -c gold_standard_sorted.vcf > gold_standard_sorted.vcf.gz
tabix -p vcf gold_standard_sorted.vcf.gz

conda activate bio_tools
bcftools reheader \
  -f "/data/chenkejin/cue-master/GRCh38.d1.vd1.fa.fai" \
  gold_standard_sorted.vcf.gz \
  -o gold_standard_ready.vcf.gz

tabix -p vcf gold_standard_ready.vcf.gz

conda activate truvari_v4

truvari评估，加上--passonly参数，获得filtered结果

In [ ]:
truvari bench \
  -b "/data/chenkejin/cue-master/gold_standard_ready.vcf.gz" \
  -c "/data/chenkejin/cue-master/result/WGS_EA_1/T-N-50/fp.vcf.gz" \
  -o "/data/chenkejin/cue-master/result/WGS_EA_1/filtered-50" \
  -f /data/share/Genomics_datasets/HCC1395/reference_genome/GRCh38/GRCh38.d1.vd1.fa \
  --passonly \
  --refdist 500 \
  --pctseq 0 \
  --pctsize 0.7 \
  --pctovl 0 \
  --sizemin 50 \
  --sizemax -1 \
  --bnddist 100

去掉--passonly参数，获得unfiltered结果

In [ ]:
truvari bench \
  -b "/data/chenkejin/cue-master/gold_standard_ready.vcf.gz" \
  -c "/data/chenkejin/cue-master/result/WGS_EA_1/T-N-50/fp.vcf.gz" \
  -o "/data/chenkejin/cue-master/result/WGS_EA_1/unfiltered-50" \
  -f /data/share/Genomics_datasets/HCC1395/reference_genome/GRCh38/GRCh38.d1.vd1.fa \
  --refdist 500 \
  --pctseq 0 \
  --pctsize 0.7 \
  --pctovl 0 \
  --sizemin 50 \
  --sizemax -1 \
  --bnddist 100

用 "/data/chenkejin/cue-master/calc_sv_scores.py" 脚本自动化提取filtered和unfiltered的DEL、DUP、INV、INS、TRA数据，并计算得分情况。

代码如下：

In [ ]:
# -*- coding: gbk -*-
import os
import gzip

def count_svtype(vcf_file, target_type):
    """
    读取 VCF 文件并统计特定 SV 类型的数量（支持 .vcf 和 .vcf.gz）
    """
    # 自动处理文件后缀，兼容 .vcf 和 .vcf.gz
    if not os.path.exists(vcf_file):
        alt_file = vcf_file[:-3] if vcf_file.endswith('.gz') else vcf_file + '.gz'
        if os.path.exists(alt_file):
            vcf_file = alt_file
        else:
            return 0 # 如果文件真不存在，返回 0

    count = 0
    open_func = gzip.open if vcf_file.endswith('.gz') else open
    
    # 针对 TRA (易位) 的特殊处理：同时匹配 TRA 和 BND
    types_to_check = [target_type]
    if target_type == 'TRA':
        types_to_check.extend(['BND'])
        
    with open_func(vcf_file, 'rt') as f:
        for line in f:
            if line.startswith('#'):
                continue
            
            parts = line.split('\t')
            if len(parts) < 8:
                continue
                
            alt = parts[4].upper()
            info = parts[7].upper()
            
            # 匹配逻辑
            is_match = False
            for t in types_to_check:
                # 匹配 INFO 里的 SVTYPE= 或者 ALT 里的 <DEL> 格式
                if f"SVTYPE={t}" in info or f"<{t}>" in alt:
                    is_match = True
                    break
                # 针对 BND (易位) 的标准断点符号匹配，如 A]chr2:1000]
                if target_type == 'TRA' and ('[' in alt or ']' in alt):
                    is_match = True
                    break
            
            if is_match:
                count += 1
                
    return count

def evaluate_directory(dir_path, dir_name):
    """
    计算特定 Truvari 输出目录下的 Precision, Recall 和 F1，并将结果保存到本地
    """
    # 新增：定义一个辅助函数，同时打印并收集文本
    output_lines = []
    def log_and_print(text):
        print(text)
        output_lines.append(text)

    log_and_print(f"\n{'='*70}")
    log_and_print(f"?? 评估报告: {dir_name}")
    log_and_print(f"{'='*70}")
    log_and_print(f"| {'SV_TYPE':<8} | {'TP':<5} | {'FP':<5} | {'FN':<5} | {'Precision':<9} | {'Recall':<9} | {'F1_Score':<9} |")
    log_and_print(f"|{'-'*10}|{'-'*7}|{'-'*7}|{'-'*7}|{'-'*11}|{'-'*11}|{'-'*11}|")
    
    for svtype in ['DEL', 'DUP', 'INV', 'INS', 'TRA']:
        # Truvari 输出的核心文件
        tp_comp_file = os.path.join(dir_path, "tp-comp.vcf.gz")
        fp_file = os.path.join(dir_path, "fp.vcf.gz")
        fn_file = os.path.join(dir_path, "fn.vcf.gz")
        
        # 统计数量
        tp = count_svtype(tp_comp_file, svtype)
        fp = count_svtype(fp_file, svtype)
        fn = count_svtype(fn_file, svtype)
        
        # 计算核心指标 (分母为0时设为0)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        
        # 打印并收集格式化结果
        log_and_print(f"| {svtype:<8} | {tp:<5} | {fp:<5} | {fn:<5} | {precision:<9.3f} | {recall:<9.3f} | {f1:<9.3f} |")

    # 新增：将收集到的报告写入当前目录下的 evaluation_report.txt 文件中
    if os.path.exists(dir_path):
        report_path = os.path.join(dir_path, "evaluation_report.txt")
        with open(report_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(output_lines) + '\n')
        print(f"> 报告已成功保存至: {report_path}")

if __name__ == "__main__":
    # 配置你的两个 Truvari 输出目录
    filtered_dir = "/data/chenkejin/cue-master/result/WGS_EA_1/filtered-4000"
    unfiltered_dir = "/data/chenkejin/cue-master/result/WGS_EA_1/unfiltered-4000"
    
    evaluate_directory(filtered_dir, "严格模式 (Filtered - PASS Only)")
    evaluate_directory(unfiltered_dir, "放宽极限模式 (Unfiltered - ALL Variants)")
    print(f"\n{'='*70}\n")

# 六、集群

In [ ]:
ln -s /mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams/1866277N/1866277N.sorted.markdup.BQSR.bam /mnt/home/ygjx/chenkejin/cue/cue-master/data_links/my_local_1866277N.bam

In [ ]:
cp /mnt/home/ygjx/chenkejin/share_group_folder_ygjx/PDAC_WGS/wgs_197_bams/1866277N/1866277N.sorted.markdup.BQSR.bai /mnt/home/ygjx/chenkejin/cue/cue-master/data_links/my_local_1866277N.bai

## config/1866277N_data.yaml

In [ ]:
bam: "/mnt/home/ygjx/chenkejin/cue/cue-master/data_links/my_local_1866277N.bam"
fai: "/mnt/home/ygjx/chenkejin/cue/cue-master/Homo_sapiens_assembly38.fasta.fai"
chr_names: null

## config/1866277N_model.yaml

In [ ]:
model_path: "/mnt/home/ygjx/chenkejin/cue/cue-master/data/models/cue.v2.pt"
out_dir: "/mnt/home/ygjx/chenkejin/cue/cue-master/output_1866277N/"
gpu_ids: [0]
n_cpus: 24
batch_size: 12

## run_cue.sh

In [ ]:
#!/bin/bash
#SBATCH --job-name=Cue_WGS
#SBATCH --partition=cu             
#SBATCH --nodes=1                  
#SBATCH --cpus-per-task=32         # 申请 32 核（完美切分大节点）
#SBATCH --mem=120G                 # 申请 120GB 内存（对 773G 来说九牛一毛，但对我们绝对够用）
#SBATCH --output=cue_1866277N.log  
#SBATCH --error=cue_1866277N.log   

# 激活环境
source /mnt/home/ygjx/chenkejin/anaconda3/bin/activate cue_env

# 防断链与无头画图魔法
export PYTHONPATH=${PYTHONPATH}:$(pwd)
export MPLBACKEND=Agg

# 打印一下实际分配的核数，然后开跑
echo "任务开始，系统分配的核数是：$(nproc)"
python engine/call.py --data_config config/1866277N_data.yaml --model_config config/1866277N_model.yaml
echo "正在转移 VCF 结果文件..."
# 将结果挪到你真正想要的输出目录，并顺便改个带有样本名的名字防混淆
mv config/reports/svs.vcf /mnt/home/ygjx/chenkejin/cue/cue-master/cue_197_output/1866277N_raw.vcfecho "全部任务完美结束！"

### 在“/mnt/home/ygjx/chenkejin/cue/cue-master/”目录下输入 sbatch run_cue.sh 提交作业

### tail -f cue_1866277N.log 实时查看日志